# manga-lens: fine-tune nhận dạng nhân vật (face / body)
Bật trước: **Internet On**, **GPU T4**, Secret `HF_TOKEN` (Attached). Mỗi phiên mới phải chạy lại từ đầu.
Quy trình: train trên 87 bộ **train**, chọn epoch/cấu hình trên **val**, rồi chạy **test đúng một lần** cho cấu hình đã chọn.
Ba script mới (`extract_characters.py`, `eval_cluster.py`, `train_reid.py`) được ghi thẳng từ notebook nên không cần push trước.

In [ ]:
!pip -q install huggingface_hub
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## 1. Tải Manga109

In [ ]:
!hf download hal-utokyo/Manga109 Manga109_released_2026_05_21.zip --repo-type dataset --local-dir /kaggle/working/m109
!cd /kaggle/working/m109 && unzip -q Manga109_released_2026_05_21.zip && rm Manga109_released_2026_05_21.zip
!ls /kaggle/working/m109

## 2. Code (clone + 3 script mới) và nhãn nhân vật
Nếu `ls` cho thấy `images/` và `annotations/` ở thư mục con khác, sửa `ROOT`.

In [ ]:
!git clone https://github.com/CPhuongg/manga-lens.git

In [ ]:
%%writefile manga-lens/scripts/extract_characters.py
"""Trích nhãn nhân vật (face/body) từ XML Manga109 -> data/characters.json.

Converter YOLO bỏ qua thuộc tính `character`; script này giữ lại, kèm split theo bộ
(đọc `data/yolo/{split}_titles.txt`). Id nhân vật chỉ có nghĩa TRONG từng bộ.

Dùng:
    python scripts/extract_characters.py --root "D:/MINI APP/dataset/Manga109_released_2026_05_21"
"""
import argparse
import json
import re
import xml.etree.ElementTree as ET
from pathlib import Path

# nhân vật nền gộp chung nhiều người -> không phải 1 danh tính
OTHER = re.compile(r"^(others?|その他|モブ.*|群衆.*|通行人.*)$", re.I)


def parse_book(xml_path):
    root = ET.parse(xml_path).getroot()
    chars = {c.get("id"): c.get("name") or "" for c in root.iter("character")}
    items = []
    for pg in root.iter("page"):
        for el in pg:
            if el.tag not in ("face", "body"):
                continue
            items.append({
                "page": int(pg.get("index")), "w": int(pg.get("width")), "h": int(pg.get("height")),
                "cls": el.tag, "id": el.get("id"), "char": el.get("character"),
                "box": [int(el.get(k)) for k in ("xmin", "ymin", "xmax", "ymax")],
            })
    return chars, items


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True, type=Path)
    ap.add_argument("--yolo", default=Path("data/yolo"), type=Path, help="chứa {split}_titles.txt")
    ap.add_argument("--out", default=Path("data/characters.json"), type=Path)
    args = ap.parse_args()

    split_of = {t: s for s in ("train", "val", "test")
                for t in (args.yolo / f"{s}_titles.txt").read_text().split()}
    books = {}
    for p in sorted((args.root / "annotations").glob("*.xml")):
        chars, items = parse_book(p)
        books[p.stem] = {
            "split": split_of[p.stem],
            "characters": {i: {"name": n, "other": bool(OTHER.match(n))} for i, n in chars.items()},
            "items": items,
        }
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(books, ensure_ascii=False))
    n = sum(len(b["items"]) for b in books.values())
    print(f"{len(books)} bộ, {n} box face/body -> {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile manga-lens/scripts/eval_cluster.py
"""Đo gom nhân vật TRONG TỪNG BỘ từ embedding (scripts/embed_chars.py).

Clustering: agglomerative average-linkage, khoảng cách cosine, cắt theo ngưỡng. Ngưỡng chọn trên val
(tối đa BCubed F1 trung bình theo bộ), rồi áp nguyên cho test. Bỏ nhân vật 'Other' khỏi tập đo.
Metric (mỗi bộ, rồi trung bình): pairwise P/R/F1, BCubed F1, ARI, NMI, số cụm vs số nhân vật thật,
retrieval Recall@1 (cùng nhân vật, loại chính nó) và mAP.

Dùng:
    python scripts/eval_cluster.py --feat face            # face | body | both
    python scripts/eval_cluster.py --feat both --min-count 2
"""
import argparse
import json
from pathlib import Path

import numpy as np


def linkage(D):
    """Average-linkage trên ma trận khoảng cách D (n,n): trả list (khoảng cách, i, j) theo thứ tự gộp.
    Average-linkage đơn điệu nên cắt tại ngưỡng bất kỳ = áp các phép gộp có khoảng cách <= ngưỡng."""
    n = len(D)
    D = D.astype(np.float64).copy()
    np.fill_diagonal(D, np.inf)
    size = np.ones(n)
    merges = []
    for _ in range(n - 1):
        k = np.argmin(D)
        i, j = divmod(k, n)
        merges.append((D[i, j], i, j))
        new = (D[i] * size[i] + D[j] * size[j]) / (size[i] + size[j])
        D[i, :] = new; D[:, i] = new; D[i, i] = np.inf
        D[j, :] = np.inf; D[:, j] = np.inf
        size[i] += size[j]
    return merges


def cut(merges, n, thr):
    label = np.arange(n)
    for d, i, j in merges:
        if d > thr:
            break
        label[label == j] = i
    return label


def contingency(a, b):
    ua, ia = np.unique(a, return_inverse=True)
    ub, ib = np.unique(b, return_inverse=True)
    M = np.zeros((len(ua), len(ub)))
    np.add.at(M, (ia, ib), 1)
    return M


def metrics(true, pred):
    M = contingency(true, pred)
    n = M.sum()
    comb = lambda x: x * (x - 1) / 2
    tp = comb(M).sum()
    p_pairs, t_pairs = comb(M.sum(0)).sum(), comb(M.sum(1)).sum()
    P = tp / p_pairs if p_pairs else 1.0
    R = tp / t_pairs if t_pairs else 1.0
    F = 2 * P * R / (P + R) if P + R else 0.0
    # BCubed
    bp = (M ** 2 / M.sum(0, keepdims=True)).sum() / n
    br = (M ** 2 / M.sum(1, keepdims=True)).sum() / n
    bf = 2 * bp * br / (bp + br) if bp + br else 0.0
    exp = p_pairs * t_pairs / comb(n)
    ari = (tp - exp) / (0.5 * (p_pairs + t_pairs) - exp) if (0.5 * (p_pairs + t_pairs) - exp) else 1.0
    pi, pj = M.sum(1) / n, M.sum(0) / n
    nz = M > 0
    mi = (M[nz] / n * np.log(M[nz] * n / (M.sum(1)[:, None] * M.sum(0)[None])[nz])).sum()
    h = lambda p: -(p * np.log(p)).sum()
    nmi = mi / ((h(pi) + h(pj)) / 2) if h(pi) + h(pj) else 1.0
    return dict(P=P, R=R, F=F, bcF=bf, ARI=ari, NMI=nmi)


def retrieval(E, y):
    S = E @ E.T
    np.fill_diagonal(S, -np.inf)
    order = np.argsort(-S, axis=1)
    r1, aps = [], []
    for i in range(len(y)):
        rel = (y[order[i]] == y[i])[:-1]  # bỏ chính nó (nằm cuối do -inf)
        if rel.sum() == 0:
            continue
        r1.append(rel[0])
        ranks = np.where(rel)[0] + 1
        aps.append((np.arange(1, len(ranks) + 1) / ranks).mean())
    return float(np.mean(r1)), float(np.mean(aps))


EMB_DIR = Path("data/emb")


def load(split, tag, feat, chars, min_count, part):
    z = np.load(EMB_DIR / f"{tag}_{split}.npz")
    d = int(z["dim_half"])
    emb = z["emb"].astype(np.float32)
    emb = emb[:, :d] if part == "cls" else emb[:, d:] if part == "mean" else emb
    books = {}
    for b in np.unique(z["book"]):
        sel = z["book"] == b
        idx, cls, e = z["idx"][sel], z["cls"][sel], emb[sel]
        cdef = chars[str(b)]["characters"]
        items = chars[str(b)]["items"]
        lab = np.array([items[i]["char"] for i in idx])
        keep = np.array([not cdef[c]["other"] for c in lab])
        if feat != "both":
            keep &= cls == feat
        e, lab = e[keep], lab[keep]
        e = e / np.linalg.norm(e, axis=1, keepdims=True).clip(1e-8)
        ids, cnt = np.unique(lab, return_counts=True)
        ok = np.isin(lab, ids[cnt >= min_count])
        if ok.sum() >= 5:
            books[str(b)] = (e[ok], lab[ok])
    return books


def dendro(books):
    return {b: linkage(1 - E @ E.T) for b, (E, y) in books.items()}


def run(books, trees, thr):
    rows = []
    for b, (E, y) in books.items():
        pred = cut(trees[b], len(y), thr)
        m = metrics(y, pred)
        m["k_pred"], m["k_true"] = len(np.unique(pred)), len(np.unique(y))
        rows.append((b, m))
    return rows


def mean(rows, key):
    return float(np.mean([m[key] for _, m in rows]))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--chars", default=Path("data/characters.json"), type=Path)
    ap.add_argument("--tag", default="dinov2-small")
    ap.add_argument("--feat", default="face", choices=["face", "body", "both"])
    ap.add_argument("--part", default="both", choices=["cls", "mean", "both"])
    ap.add_argument("--min-count", type=int, default=2, help="chỉ giữ nhân vật có ≥ N box")
    ap.add_argument("--emb-dir", default=Path("data/emb"), type=Path)
    ap.add_argument("--no-test", action="store_true", help="chỉ đánh giá val (chọn tham số), không đụng test")
    ap.add_argument("--thrs", type=float, nargs="+", default=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
    args = ap.parse_args()
    global EMB_DIR
    EMB_DIR = args.emb_dir
    chars = json.loads(args.chars.read_text(encoding="utf-8"))

    val = load("val", args.tag, args.feat, chars, args.min_count, args.part)
    test = {} if args.no_test else load("test", args.tag, args.feat, chars, args.min_count, args.part)
    print(f"feat={args.feat} part={args.part} min_count={args.min_count}: val {len(val)} bộ "
          f"({sum(len(v[1]) for v in val.values())} box), test {len(test)} bộ ({sum(len(v[1]) for v in test.values())} box)")
    best = (-1, None)
    vtrees = dendro(val)
    for t in args.thrs:
        r = run(val, vtrees, t)
        print(f"  val thr {t:.2f}: bcF {mean(r,'bcF'):.3f} pairF {mean(r,'F'):.3f} ARI {mean(r,'ARI'):.3f} "
              f"k_pred/k_true {mean(r,'k_pred'):.1f}/{mean(r,'k_true'):.1f}")
        if mean(r, "bcF") > best[0]:
            best = (mean(r, "bcF"), t)
    thr = best[1]
    print(f"-> chọn ngưỡng {thr} theo val")
    if args.no_test:
        return
    rows = run(test, dendro(test), thr)
    print(f"{'bộ':24s} {'P':>5s} {'R':>5s} {'pairF':>6s} {'bcF':>5s} {'ARI':>5s} {'NMI':>5s} {'k_pred':>6s} {'k_true':>6s} {'R@1':>5s} {'mAP':>5s}")
    r1s, maps = [], []
    for b, m in rows:
        E, y = test[b]
        r1, mp = retrieval(E, y)
        r1s.append(r1); maps.append(mp)
        print(f"{b:24s} {m['P']:5.3f} {m['R']:5.3f} {m['F']:6.3f} {m['bcF']:5.3f} {m['ARI']:5.3f} {m['NMI']:5.3f} "
              f"{m['k_pred']:6d} {m['k_true']:6d} {r1s[-1]:5.3f} {maps[-1]:5.3f}")
    print(f"{'TRUNG BÌNH (theo bộ)':24s} {mean(rows,'P'):5.3f} {mean(rows,'R'):5.3f} {mean(rows,'F'):6.3f} "
          f"{mean(rows,'bcF'):5.3f} {mean(rows,'ARI'):5.3f} {mean(rows,'NMI'):5.3f} {mean(rows,'k_pred'):6.1f} "
          f"{mean(rows,'k_true'):6.1f} {np.mean(r1s):5.3f} {np.mean(maps):5.3f}")
    # baseline ngẫu nhiên: nhãn xáo trộn, cùng số cụm thật
    rng = np.random.default_rng(0)
    rb = []
    for b, (E, y) in test.items():
        rb.append(metrics(y, rng.permutation(y))["bcF"])
    print(f"baseline xáo nhãn (bcF): {np.mean(rb):.3f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile manga-lens/scripts/train_reid.py
"""Fine-tune DINOv2-small bằng metric learning (SupCon) để gom nhân vật theo face hoặc body.

Nhãn = (bộ, nhân vật) lấy từ data/characters.json (scripts/extract_characters.py); bỏ 'Other'.
Train trên split train, chọn epoch/tham số theo val (retrieval mAP mỗi epoch, rồi eval_cluster --no-test).
Test CHỈ đụng khi chạy --eval-only (một lần, cho cấu hình đã chọn trên val).

Crop được cache ra uint8 (face 126x126, body 224x112) trong --cache; ảnh đọc từ <root>/images/<bộ>/<trang>.jpg.

    python scripts/train_reid.py --root <Manga109> --cls face --tag f_a --epochs 10
    python scripts/eval_cluster.py --chars /kaggle/working/characters.json --tag f_a --feat face --emb-dir out/reid --no-test        # val
    python scripts/train_reid.py --root <Manga109> --cls face --tag f_a --eval-only          # ghi emb val+test
    python scripts/eval_cluster.py --chars /kaggle/working/characters.json --tag f_a --feat face --emb-dir out/reid                  # test
"""
import argparse
import json
import sys
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from transformers import AutoModel

sys.path.insert(0, str(Path(__file__).parent))
from eval_cluster import retrieval  # noqa: E402

SIZE = {"face": (126, 126), "body": (224, 112)}  # (H, W), chia hết cho patch 14
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


# ---------------------------------------------------------------- dữ liệu
def build_crops(books, split, root, cls, cache, margin=0.1, max_books=0):
    """Cắt + resize toàn bộ box `cls` của split -> memmap uint8 + meta. Có cache nên chạy lại rất nhanh."""
    cache.mkdir(parents=True, exist_ok=True)
    f_img, f_meta = cache / f"{cls}_{split}.npy", cache / f"{cls}_{split}.json"
    if f_img.exists() and f_meta.exists():
        return np.load(f_img, mmap_mode="r"), json.loads(f_meta.read_text(encoding="utf-8"))
    H, W = SIZE[cls]
    titles = [t for t, b in books.items() if b["split"] == split]
    if max_books:
        titles = titles[:max_books]
    n = sum(1 for t in titles for it in books[t]["items"] if it["cls"] == cls)
    arr = np.lib.format.open_memmap(f_img, mode="w+", dtype=np.uint8, shape=(n, H, W, 3))
    meta, k = [], 0
    t0 = time.time()
    for ti, title in enumerate(titles):
        book = books[title]
        by_page = defaultdict(list)
        for i, it in enumerate(book["items"]):
            if it["cls"] == cls:
                by_page[it["page"]].append(i)
        for page, ids in by_page.items():
            p = root / "images" / title / f"{page:03d}.jpg"
            if not p.exists():
                continue
            img = Image.open(p).convert("RGB")
            for i in ids:
                it = book["items"][i]
                x1, y1, x2, y2 = it["box"]
                w, h = x2 - x1, y2 - y1
                box = (max(0, int(x1 - margin * w)), max(0, int(y1 - margin * h)),
                       min(img.width, int(x2 + margin * w)), min(img.height, int(y2 + margin * h)))
                if box[2] - box[0] < 2 or box[3] - box[1] < 2:
                    continue
                arr[k] = np.asarray(img.crop(box).resize((W, H), Image.BILINEAR, reducing_gap=2.0))
                meta.append({"book": title, "idx": i, "char": it["char"],
                             "other": book["characters"][it["char"]]["other"]})
                k += 1
        print(f"  [{split}/{cls}] {ti + 1}/{len(titles)} {title} ({k} crop, {time.time() - t0:.0f}s)", flush=True)
    arr.flush()
    del arr
    arr = np.load(f_img, mmap_mode="r")[:k]  # k <= n nếu có crop bị bỏ
    f_meta.write_text(json.dumps(meta), encoding="utf-8")
    return arr, meta


# ---------------------------------------------------------------- mô hình
class ReID(nn.Module):
    def __init__(self, name="facebook/dinov2-small", emb=256, unfreeze=4):
        super().__init__()
        self.bb = AutoModel.from_pretrained(name)
        d = self.bb.config.hidden_size
        for p in self.bb.parameters():
            p.requires_grad = False
        layers = self.bb.encoder.layer
        for blk in layers[len(layers) - unfreeze:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in self.bb.layernorm.parameters():
            p.requires_grad = unfreeze > 0
        self.head = nn.Linear(2 * d, emb)

    def forward(self, x):
        o = self.bb(pixel_values=x).last_hidden_state
        f = torch.cat([o[:, 0], o[:, 1:].mean(1)], dim=1)
        return F.normalize(self.head(f), dim=-1)


def prep(xu8, dev, aug):
    """uint8 (B,H,W,3) -> tensor chuẩn hoá trên GPU; aug = lật, zoom/dịch, sáng/tương phản, xám."""
    x = torch.from_numpy(np.ascontiguousarray(xu8)).to(dev).permute(0, 3, 1, 2).float() / 255
    B, _, H, W = x.shape
    if aug:
        s = torch.empty(B, device=dev).uniform_(0.8, 1.0)
        flip = (torch.rand(B, device=dev) < 0.5).float() * 2 - 1
        tx = (torch.rand(B, device=dev) * 2 - 1) * (1 - s)
        ty = (torch.rand(B, device=dev) * 2 - 1) * (1 - s)
        theta = torch.zeros(B, 2, 3, device=dev)
        theta[:, 0, 0] = s * flip; theta[:, 0, 2] = tx
        theta[:, 1, 1] = s; theta[:, 1, 2] = ty
        x = F.grid_sample(x, F.affine_grid(theta, x.shape, align_corners=False),
                          padding_mode="border", align_corners=False)
        b = torch.empty(B, 1, 1, 1, device=dev).uniform_(-0.15, 0.15)
        c = torch.empty(B, 1, 1, 1, device=dev).uniform_(0.8, 1.2)
        m = x.mean((1, 2, 3), keepdim=True)
        x = ((x - m) * c + m + b).clamp(0, 1)
        g = (torch.rand(B, 1, 1, 1, device=dev) < 0.2).float()
        x = g * x.mean(1, keepdim=True).expand(-1, 3, -1, -1) + (1 - g) * x
    mean = torch.tensor(MEAN, device=dev).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=dev).view(1, 3, 1, 1)
    return (x - mean) / std


@torch.no_grad()
def embed_all(model, arr, dev, bs=256):
    model.eval()
    out = []
    for i in range(0, len(arr), bs):
        with torch.autocast(dev, dtype=torch.float16, enabled=dev == "cuda"):
            out.append(model(prep(arr[i:i + bs], dev, False)).float().cpu())
    return torch.cat(out).numpy()


def supcon(z, y, temp):
    sim = z @ z.T / temp
    eye = torch.eye(len(z), dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(eye, -1e9)
    pos = (y[:, None] == y[None]) & ~eye
    logp = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    n_pos = pos.sum(1)
    ok = n_pos > 0
    return -((logp * pos).sum(1)[ok] / n_pos[ok]).mean()


def val_retrieval(emb, meta, min_count=2):
    """Recall@1 / mAP trung bình theo bộ (bỏ Other, giữ nhân vật có >= min_count box) — như eval_cluster."""
    by_book = defaultdict(list)
    for i, m in enumerate(meta):
        if not m["other"]:
            by_book[m["book"]].append(i)
    r1s, maps = [], []
    for b, ids in by_book.items():
        y = np.array([meta[i]["char"] for i in ids])
        u, cnt = np.unique(y, return_counts=True)
        keep = np.isin(y, u[cnt >= min_count])
        if keep.sum() < 5:
            continue
        r1, mp = retrieval(emb[np.array(ids)[keep]], y[keep])
        r1s.append(r1); maps.append(mp)
    return float(np.mean(r1s)), float(np.mean(maps))


def save_emb(path, emb, meta, cls):
    np.savez_compressed(path, book=np.array([m["book"] for m in meta]), idx=np.array([m["idx"] for m in meta]),
                        cls=np.array([cls] * len(meta)), emb=emb.astype(np.float16), dim_half=emb.shape[1] // 2)


# ---------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True, type=Path, help="thư mục Manga109 (có images/)")
    ap.add_argument("--chars", default=Path("data/characters.json"), type=Path)
    ap.add_argument("--cls", default="face", choices=["face", "body"])
    ap.add_argument("--tag", required=True)
    ap.add_argument("--out", default=Path("out/reid"), type=Path)
    ap.add_argument("--cache", default=Path("data/crops"), type=Path)
    ap.add_argument("--model", default="facebook/dinov2-small")
    ap.add_argument("--emb", type=int, default=256)
    ap.add_argument("--unfreeze", type=int, default=4, help="số block cuối của backbone được train")
    ap.add_argument("--lr-bb", type=float, default=2e-5)
    ap.add_argument("--lr-head", type=float, default=1e-3)
    ap.add_argument("--temp", type=float, default=0.1)
    ap.add_argument("--P", type=int, default=32, help="số nhân vật mỗi batch")
    ap.add_argument("--K", type=int, default=4, help="số mẫu mỗi nhân vật")
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--iters", type=int, default=0, help="bước mỗi epoch (0 = n_mẫu/(P*K))")
    ap.add_argument("--max-books", type=int, default=0, help="debug: giới hạn số bộ mỗi split")
    ap.add_argument("--eval-only", action="store_true", help="nạp best.pt, ghi embedding val+TEST")
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    torch.manual_seed(args.seed); np.random.seed(args.seed)
    rng = np.random.default_rng(args.seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    books = json.loads(args.chars.read_text(encoding="utf-8"))
    out = args.out / args.tag
    out.mkdir(parents=True, exist_ok=True)
    model = ReID(args.model, args.emb, args.unfreeze).to(dev)

    splits = ["val", "test"] if args.eval_only else ["train", "val"]
    data = {s: build_crops(books, s, args.root, args.cls, args.cache, max_books=args.max_books) for s in splits}

    if args.eval_only:
        model.load_state_dict(torch.load(out / "best.pt", map_location=dev))
        for s in ("val", "test"):
            arr, meta = data[s]
            emb = embed_all(model, arr, dev)
            save_emb(args.out / f"{args.tag}_{s}.npz", emb, meta, args.cls)
            print(s, "->", args.out / f"{args.tag}_{s}.npz", emb.shape)
        return

    # --- danh sách mẫu train theo nhân vật
    tr_arr, tr_meta = data["train"]
    tr_arr = np.ascontiguousarray(tr_arr)  # nạp vào RAM
    groups = defaultdict(list)
    for i, m in enumerate(tr_meta):
        if not m["other"]:
            groups[(m["book"], m["char"])].append(i)
    groups = {k: np.array(v) for k, v in groups.items() if len(v) >= 2}
    keys = list(groups)
    w = np.array([np.sqrt(len(groups[k])) for k in keys]); w /= w.sum()
    n_samples = sum(len(v) for v in groups.values())
    iters = args.iters or max(1, n_samples // (args.P * args.K))
    print(f"train: {len(keys)} nhân vật, {n_samples} crop, {iters} bước/epoch", flush=True)

    params = [{"params": [p for p in model.bb.parameters() if p.requires_grad], "lr": args.lr_bb},
              {"params": model.head.parameters(), "lr": args.lr_head}]
    opt = torch.optim.AdamW(params, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[args.lr_bb, args.lr_head],
                                                total_steps=args.epochs * iters, pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=dev == "cuda")
    va_arr, va_meta = data["val"]

    # mốc 0: chưa fine-tune (head ngẫu nhiên) — chỉ để biết val ban đầu
    best, log = -1.0, []
    for ep in range(1, args.epochs + 1):
        model.train()
        t0, tot = time.time(), 0.0
        for it in range(iters):
            sel = rng.choice(len(keys), size=args.P, replace=False, p=w)
            idx, lab = [], []
            for j, k in enumerate(sel):
                g = groups[keys[k]]
                pick = rng.choice(g, size=args.K, replace=len(g) < args.K)
                idx.extend(pick.tolist()); lab.extend([j] * args.K)
            order = np.argsort(idx)  # đọc tuần tự hơn; nhãn đi theo
            idx = np.array(idx)[order]; lab = torch.tensor(np.array(lab)[order], device=dev)
            x = prep(tr_arr[idx], dev, True)
            with torch.autocast(dev, dtype=torch.float16, enabled=dev == "cuda"):
                z = model(x)
            loss = supcon(z.float(), lab, args.temp)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()
        emb = embed_all(model, va_arr, dev)
        r1, mp = val_retrieval(emb, va_meta)
        log.append({"epoch": ep, "loss": tot / iters, "val_R1": r1, "val_mAP": mp})
        print(f"epoch {ep}/{args.epochs} loss {tot / iters:.4f} val R@1 {r1:.3f} mAP {mp:.3f} ({time.time() - t0:.0f}s)", flush=True)
        if mp > best:
            best = mp
            torch.save(model.state_dict(), out / "best.pt")
        (out / "log.json").write_text(json.dumps({"args": {k: str(v) for k, v in vars(args).items()}, "log": log}))
    print(f"best val mAP {best:.3f} (epoch {max(log, key=lambda r: r['val_mAP'])['epoch']})")

    # embedding val của epoch tốt nhất để chạy eval_cluster --no-test
    model.load_state_dict(torch.load(out / "best.pt", map_location=dev))
    save_emb(args.out / f"{args.tag}_val.npz", embed_all(model, va_arr, dev), va_meta, args.cls)
    print("val embedding ->", args.out / f"{args.tag}_val.npz")


if __name__ == "__main__":
    main()

In [ ]:
import sys, os
sys.path.insert(0, "manga-lens/scripts")
from pathlib import Path
from convert_manga109 import split_titles   # cùng cách chia (seed 0, theo bộ) như lúc train detector
ROOT = "/kaggle/working/m109/Manga109_released_2026_05_21"
titles = [p.stem for p in Path(ROOT, "annotations").glob("*.xml")]
sp = split_titles(titles, seed=0)
os.makedirs("/kaggle/working/titles", exist_ok=True)
for s, t in sp.items():
    Path(f"/kaggle/working/titles/{s}_titles.txt").write_text("
".join(sorted(t)))
    print(s, len(t), "bộ")
!cd manga-lens && PYTHONUTF8=1 python scripts/extract_characters.py --root {ROOT} --yolo /kaggle/working/titles --out /kaggle/working/characters.json

## 3. Thử nhanh (2 bộ, vài bước) để kiểm tra pipeline

In [ ]:
CACHE = "/kaggle/temp/crops" if os.path.isdir("/kaggle/temp") else "/kaggle/working/crops"
COMMON = f"--root {ROOT} --chars /kaggle/working/characters.json --cache {CACHE}_smoke --out /kaggle/working/out/smoke"
!cd manga-lens && PYTHONUTF8=1 python scripts/train_reid.py {COMMON} --cls face --tag smoke --epochs 1 --iters 10 --max-books 2

## 4. Face: so vài cấu hình, chọn trên VAL
Mỗi cấu hình ~10 epoch. `CACHE` được dùng chung nên crop chỉ cắt một lần. Số theo dõi mỗi epoch là retrieval trên val;
số quyết định là `eval_cluster --no-test` ở bước 5.

In [ ]:
COMMON = f"--root {ROOT} --chars /kaggle/working/characters.json --cache {CACHE} --out /kaggle/working/out/reid"
!cd manga-lens && PYTHONUTF8=1 python scripts/train_reid.py {COMMON} --cls face --tag f_a --unfreeze 4  --lr-bb 2e-5 --epochs 10

In [ ]:
!cd manga-lens && PYTHONUTF8=1 python scripts/train_reid.py {COMMON} --cls face --tag f_b --unfreeze 12 --lr-bb 1e-5 --epochs 10

In [ ]:
!cd manga-lens && PYTHONUTF8=1 python scripts/train_reid.py {COMMON} --cls face --tag f_c --unfreeze 4  --lr-bb 2e-5 --temp 0.05 --epochs 10

## 5. Chọn trên VAL (không đụng test)

In [ ]:
for tag in ["f_a", "f_b", "f_c"]:
    print("=====", tag)
    !cd manga-lens && PYTHONUTF8=1 python scripts/eval_cluster.py --chars /kaggle/working/characters.json --tag {tag} --feat face --emb-dir /kaggle/working/out/reid --no-test | grep -E "feat=|thr|chọn"

## 6. Test — một lần, cho cấu hình đã chọn
Đặt `BEST_FACE` theo bảng val ở trên (tiêu chí: BCubed F1 val cao nhất; xem thêm số cụm so với số nhân vật thật).

In [ ]:
BEST_FACE = "f_a"   # <-- sửa sau khi xem bước 5
!cd manga-lens && PYTHONUTF8=1 python scripts/train_reid.py {COMMON} --cls face --tag {BEST_FACE} --unfreeze 4 --eval-only
!cd manga-lens && PYTHONUTF8=1 python scripts/eval_cluster.py --chars /kaggle/working/characters.json --tag {BEST_FACE} --feat face --emb-dir /kaggle/working/out/reid

## 7. Body (một cấu hình, cùng cách)
Xoá cache face trước để đỡ đầy ổ đĩa (crop body train ~9 GB).

In [ ]:
!rm -rf {CACHE}/face_*
!cd manga-lens && PYTHONUTF8=1 python scripts/train_reid.py {COMMON} --cls body --tag b_a --unfreeze 4 --lr-bb 2e-5 --epochs 10
!cd manga-lens && PYTHONUTF8=1 python scripts/eval_cluster.py --chars /kaggle/working/characters.json --tag b_a --feat body --emb-dir /kaggle/working/out/reid --no-test | grep -E "feat=|thr|chọn"

In [ ]:
!cd manga-lens && PYTHONUTF8=1 python scripts/train_reid.py {COMMON} --cls body --tag b_a --unfreeze 4 --eval-only
!cd manga-lens && PYTHONUTF8=1 python scripts/eval_cluster.py --chars /kaggle/working/characters.json --tag b_a --feat body --emb-dir /kaggle/working/out/reid

## 8. Lưu kết quả (chỉ /kaggle/working được giữ)
`out/reid/<tag>/best.pt` + `log.json`, và embedding `*.npz`.

In [ ]:
!rm -rf /kaggle/working/m109 /kaggle/working/out/smoke
!cp /kaggle/working/characters.json /kaggle/working/out/
!du -sh /kaggle/working/out/*; ls /kaggle/working/out/reid